# M5 Pipeline — GPU (Training + Eval + Submission)

Split ファイルを読み込み → LightGBM 学習 → 評価 → 提出ファイル生成

## Step 0: Setup

In [ ]:
# ============================================================
# SETUP — Colab / Local 両対応
# ============================================================
import sys, os, subprocess, time
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (14, 5)

IS_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ
print(f"Environment: {'Google Colab' if IS_COLAB else 'Local'}")

# pip install (Colab のみ)
if IS_COLAB:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           'pyarrow', 'lightgbm', 'scikit-learn'])

COMPETITION = 'm5-forecasting-accuracy'
DATA_FILES  = ['calendar.csv', 'sales_train_evaluation.csv', 'sell_prices.csv']

def has_all_files(d: Path) -> bool:
    return d.exists() and all((d / f).exists() for f in DATA_FILES)

# ============================================================
# Drive マウント (Colab のみ — 出力先に使用)
# ============================================================
_drive_ok = False
if IS_COLAB:
    _drive_ok = Path('/content/drive/MyDrive').exists()
    if not _drive_ok:
        try:
            from google.colab import drive
            drive.mount('/content/drive', force_remount=False)
            _drive_ok = True
        except Exception as _e:
            print(f'Drive mount failed: {_e}')

# ============================================================
# DATA_DIR 検出 (CSV 読み込み元)
# ============================================================
DATA_DIR = None

if IS_COLAB:
    for c in [Path(f'/content/{COMPETITION}'), Path('/content/data')]:
        if has_all_files(c):
            DATA_DIR = c
            break
    if DATA_DIR is None and _drive_ok:
        for c in [
            Path(f'/content/drive/MyDrive/kaggle/{COMPETITION}'),
            Path(f'/content/drive/MyDrive/{COMPETITION}'),
        ]:
            if has_all_files(c):
                DATA_DIR = c
                break
    if DATA_DIR is None:
        # CSV がどこにもない → ダウンロード
        DATA_DIR = Path(f'/content/{COMPETITION}')
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        _token = None
        _token_paths = [
            Path('/content/drive/MyDrive/.kaggle/access_token'),
            Path('/content/drive/MyDrive/kaggle/access_token'),
        ]
        for _tp in _token_paths:
            if _tp.exists():
                _token = _tp.read_text().strip()
                break
        if _token is None:
            try:
                from google.colab import userdata
                _token = userdata.get('KAGGLE_TOKEN')
            except Exception:
                pass
        if _token is None:
            raise RuntimeError(
                'Kaggle token が見つかりません。\n'
                'Drive に .kaggle/access_token を配置するか、\n'
                'Colab Secrets に KAGGLE_TOKEN を設定してください'
            )
        import requests, zipfile
        _url = f'https://www.kaggle.com/api/v1/competitions/data/download-all/{COMPETITION}'
        print('Downloading...')
        with requests.get(_url, headers={'Authorization': f'Bearer {_token}'},
                          stream=True, allow_redirects=True) as _r:
            _r.raise_for_status()
            _zip = DATA_DIR / f'{COMPETITION}.zip'
            with open(_zip, 'wb') as _f:
                for _chunk in _r.iter_content(1024*1024):
                    _f.write(_chunk)
        with zipfile.ZipFile(_zip) as _z:
            _z.extractall(DATA_DIR)
        _zip.unlink()
        del _token
        print('Download complete')
else:
    for c in [
        Path(f'/mnt/c/Users/hiroyuki_nakayama/src/kaggle-with-mcp/{COMPETITION}'),
        Path('.'),
    ]:
        if has_all_files(c):
            DATA_DIR = c
            break

assert DATA_DIR is not None and has_all_files(DATA_DIR), \
    f'CSV ファイルが見つかりません: {DATA_DIR}'

# ============================================================
# OUTPUT_DIR (前処理結果の書き出し先)
# ============================================================
if IS_COLAB and _drive_ok:
    OUTPUT_DIR = Path(f'/content/drive/MyDrive/kaggle/{COMPETITION}')
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
else:
    OUTPUT_DIR = DATA_DIR

print(f'DATA_DIR   : {DATA_DIR}  (CSV 読み込み元)')
print(f'OUTPUT_DIR : {OUTPUT_DIR}  (出力先)')
for f in DATA_FILES:
    print(f'  {f:<40} {(DATA_DIR/f).stat().st_size/1e6:6.1f} MB')

## Load: 変数復元

In [ ]:
# ============================================================
# Load: split ファイルから変数を復元
# ============================================================
import pyarrow.parquet as pq

# 定数 (Phase 2 と同一)
PRED_HORIZON = 28
META_COLS = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
TARGET = 'sales'
TARGET_EPS = 1.0
RATIO_TARGET_CATS = set()  # 空 = 全カテゴリ raw target
CAT_NAMES = {0: 'FOODS', 1: 'HOBBIES', 2: 'HOUSEHOLD'}
PAYDAY_DAYS = {14, 15, 16, 28, 29, 30, 31}
PAYDAY_EXACT = {15, 28, 29, 30, 31}

OUT_PATH    = OUTPUT_DIR / 'df_features.parquet'
TRAIN_X_PATH   = OUTPUT_DIR / 'train_X.dat'
TRAIN_Y_PATH   = OUTPUT_DIR / 'train_y.dat'
TRAIN_CAT_PATH = OUTPUT_DIR / 'train_cat.dat'
VAL_PATH       = OUTPUT_DIR / 'val.parquet'
EVAL_PATH      = OUTPUT_DIR / 'eval.parquet'

# FEATURES: parquet schema から復元
pf = pq.ParquetFile(OUT_PATH)
all_cols = [f.name for f in pf.schema_arrow]
DROP_COLS = set(META_COLS + ['d', 'date', 'sales', 'wm_yr_wk', 'd_num',
                             'snap_CA', 'snap_TX', 'snap_WI'])
FEATURES = [c for c in all_cols if c not in DROP_COLS]
del pf

# n_train: memmap サイズから逆算
n_train = os.path.getsize(TRAIN_X_PATH) // (len(FEATURES) * 4)

print(f'Features: {len(FEATURES)}')
print(f'Train: {n_train:,} rows')
print(f'Val: {pq.ParquetFile(VAL_PATH).metadata.num_rows:,} rows')
print(f'Eval: {pq.ParquetFile(EVAL_PATH).metadata.num_rows:,} rows')
print(f'Target: 全カテゴリ raw sales + tweedie')

for p in [TRAIN_X_PATH, TRAIN_Y_PATH, TRAIN_CAT_PATH, VAL_PATH, EVAL_PATH]:
    assert p.exists(), f'MISSING: {p}'
print('\nAll split files OK')

## Step 5: モデル学習

In [ ]:
# ============================================================
# Step 5: LightGBM 学習 — 2モデル分割 (FOODS / non-FOODS)
# ============================================================
import lightgbm as lgb

# Train: memmap
X_train = np.memmap(str(TRAIN_X_PATH), dtype='float32', mode='r',
                    shape=(n_train, len(FEATURES)))
y_train = np.memmap(str(TRAIN_Y_PATH), dtype='float32', mode='r',
                    shape=(n_train,))
cat_train = np.memmap(str(TRAIN_CAT_PATH), dtype='int8', mode='r',
                      shape=(n_train,))

# Val: row_group ごとに読んで concat (小さいので許容)
pf_val = pq.ParquetFile(VAL_PATH)
val_parts = []
for i in range(pf_val.metadata.num_row_groups):
    val_parts.append(pf_val.read_row_group(i).to_pandas())
df_val = pd.concat(val_parts, ignore_index=True)
del val_parts, pf_val
df_val[FEATURES] = df_val[FEATURES].fillna(0)

_device = 'gpu' if os.environ.get('COLAB_GPU') or os.path.exists('/dev/nvidia0') else 'cpu'

# === 2モデル構成: FOODS / non-FOODS ===
# FOODS: feature_fraction=0.7 で価格系特徴量の選択確率を上げる
# non-FOODS: HOBBIES + HOUSEHOLD を統合 (48K の HOUSEHOLD を保護)

MODEL_GROUPS = {
    'FOODS': {
        'cat_ids': [0],
        'lgb_params': {
            'objective'         : 'tweedie',
            'metric'            : 'rmse',
            'num_leaves'        : 127,
            'learning_rate'     : 0.03,
            'feature_fraction'  : 0.7,    # ← 0.8→0.7: 価格系特徴量の学習機会を増やす
            'bagging_fraction'  : 0.8,
            'bagging_freq'      : 1,
            'min_child_samples' : 20,
            'device'            : _device,
            'n_jobs'            : -1,
            'seed'              : 42,
            'verbosity'         : -1,
            'two_round'         : True,
            'max_bin'           : 127,
        },
        'num_boost_round': 3000,
        'early_stopping': 100,
    },
    'NON_FOODS': {
        'cat_ids': [1, 2],  # HOBBIES + HOUSEHOLD
        'lgb_params': {
            'objective'         : 'tweedie',
            'metric'            : 'rmse',
            'num_leaves'        : 127,
            'learning_rate'     : 0.03,
            'feature_fraction'  : 0.8,
            'bagging_fraction'  : 0.8,
            'bagging_freq'      : 1,
            'min_child_samples' : 20,
            'device'            : _device,
            'n_jobs'            : -1,
            'seed'              : 42,
            'verbosity'         : -1,
            'two_round'         : True,
            'max_bin'           : 127,
        },
        'num_boost_round': 2500,
        'early_stopping': 80,
    },
}

print(f'[INFO] Device: {_device}')
print(f'[INFO] 2-model split: FOODS (ff=0.7) / NON_FOODS (HOBBIES+HOUSEHOLD)')

models = {}
for group_name, cfg in MODEL_GROUPS.items():
    print(f'\n{"="*60}')
    print(f'Training: {group_name} (cat_ids={cfg["cat_ids"]})')
    print(f'{"="*60}')

    # Train mask: cat_id in group
    tr_mask = np.isin(cat_train, cfg['cat_ids'])
    n_cat = int(tr_mask.sum())
    tr_idx = np.where(tr_mask)[0]
    X_cat = X_train[tr_idx]
    y_cat = y_train[tr_idx]

    # Val mask
    val_mask = df_val['cat_id'].isin(cfg['cat_ids'])
    df_val_cat = df_val[val_mask]

    print(f'  train: {n_cat:,} rows, val: {len(df_val_cat):,} rows')
    print(f'  rounds: {cfg["num_boost_round"]}, early_stopping: {cfg["early_stopping"]}')
    print(f'  feature_fraction: {cfg["lgb_params"]["feature_fraction"]}')

    dtrain = lgb.Dataset(
        X_cat, label=y_cat,
        feature_name=FEATURES,
        free_raw_data=True,
    )
    dval = lgb.Dataset(
        df_val_cat[FEATURES].values, label=df_val_cat[TARGET].values,
        reference=dtrain,
        free_raw_data=True,
    )

    model = lgb.train(
        cfg['lgb_params'], dtrain,
        num_boost_round=cfg['num_boost_round'],
        valid_sets=[dtrain, dval],
        valid_names=['train', 'val'],
        callbacks=[lgb.early_stopping(cfg['early_stopping']), lgb.log_evaluation(100)],
    )
    models[group_name] = model
    print(f'  Best iteration: {model.best_iteration}')
    print(f'  Best val RMSE : {model.best_score["val"]["rmse"]:.4f}')

    del X_cat, y_cat, tr_idx, dtrain, dval, df_val_cat

del X_train, y_train, cat_train
print(f'\n{len(models)} モデル学習完了')


## Step 6: 評価 (RMSE + Feature Importance)

In [ ]:
# ============================================================
# Step 6: 評価 (RMSE + Feature Importance) — 2モデル構成
# ============================================================
from sklearn.metrics import root_mean_squared_error

# cat_id → model group mapping
def get_model_for_cat(cat_id):
    for gname, cfg in MODEL_GROUPS.items():
        if cat_id in cfg['cat_ids']:
            return models[gname]
    raise ValueError(f'No model for cat_id={cat_id}')

# Val 予測
pf_val = pq.ParquetFile(VAL_PATH)
all_true = []
all_pred = []
for i in range(pf_val.metadata.num_row_groups):
    rg = pf_val.read_row_group(i).to_pandas()
    rg[FEATURES] = rg[FEATURES].fillna(0)
    pred = np.zeros(len(rg), dtype='float32')
    for cat_id in CAT_NAMES:
        mask = (rg['cat_id'] == cat_id)
        if mask.any():
            model = get_model_for_cat(cat_id)
            pred[mask] = np.clip(model.predict(rg.loc[mask, FEATURES].values), 0, None)
    all_true.append(rg[TARGET].values)
    all_pred.append(pred)
    del rg
del pf_val

val_true = np.concatenate(all_true)
val_pred = np.concatenate(all_pred)
del all_true, all_pred

rmse = root_mean_squared_error(val_true, val_pred)
print(f'Val RMSE (全体): {rmse:.4f}')

# cat_id 別 RMSE
pf_val = pq.ParquetFile(VAL_PATH)
cat_true = {c: [] for c in CAT_NAMES}
cat_pred = {c: [] for c in CAT_NAMES}
for i in range(pf_val.metadata.num_row_groups):
    rg = pf_val.read_row_group(i).to_pandas()
    rg[FEATURES] = rg[FEATURES].fillna(0)
    for cat_id in CAT_NAMES:
        mask = (rg['cat_id'] == cat_id)
        if mask.any():
            model = get_model_for_cat(cat_id)
            cat_true[cat_id].append(rg.loc[mask, TARGET].values)
            cat_pred[cat_id].append(np.clip(model.predict(rg.loc[mask, FEATURES].values), 0, None))
    del rg
del pf_val

for cat_id, cat_name in CAT_NAMES.items():
    t = np.concatenate(cat_true[cat_id])
    p = np.concatenate(cat_pred[cat_id])
    print(f'  {cat_name:>10}: RMSE={root_mean_squared_error(t, p):.4f} (n={len(t):,})')
del cat_true, cat_pred, val_true, val_pred

# --- Feature Importance: 2モデル + カテゴリ別 (FOODS / NON_FOODS) ---
os.makedirs(str(OUTPUT_DIR / 'figures'), exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(22, 10))

for idx, (gname, model) in enumerate(models.items()):
    imp = model.feature_importance('gain').astype('float64')
    fi = (
        pd.DataFrame({'feature': FEATURES, 'importance': imp})
        .sort_values('importance', ascending=False)
    )
    ax = axes[idx]
    fi.head(25).plot.barh(x='feature', y='importance', ax=ax, legend=False,
                           color='steelblue' if idx == 0 else 'coral')
    ax.set_title(f'Feature Importance — {gname} (Top 25, gain)')
    ax.invert_yaxis()
    print(f'\n--- {gname} Top 10 ---')
    print(fi.head(10).to_string(index=False))

    # Step 12d 新特徴量の順位を確認
    new_feats = ['value_gap', 'value_gap_x_elasticity', 'deal_intensity',
                 'above_price_wall', 'price_rank_in_dept', 'price_rolling_mean_56']
    print(f'\n--- {gname}: Step 12d 新特徴量の順位 ---')
    for feat in new_feats:
        if feat in fi['feature'].values:
            rank = fi.reset_index(drop=True)
            rank = rank[rank['feature'] == feat].index[0] + 1
            imp_val = fi[fi['feature'] == feat]['importance'].values[0]
            print(f'  {feat:30s}: rank={rank:>3d}, importance={imp_val:.2e}')
        else:
            print(f'  {feat:30s}: NOT FOUND')

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'figures/feature_importance_2model.png'), dpi=100, bbox_inches='tight')
plt.show()


## Step 7: Submission 生成

In [ ]:
# ============================================================
# Step 7: Submission 生成 — 2モデル構成で予測 → pivot
# ============================================================

def predict_parquet_to_wide(parquet_path, suffix):
    pf = pq.ParquetFile(parquet_path)
    ids, d_nums, preds = [], [], []
    for i in range(pf.metadata.num_row_groups):
        rg = pf.read_row_group(i).to_pandas()
        rg[FEATURES] = rg[FEATURES].fillna(0)
        pred = np.zeros(len(rg), dtype='float32')
        for cat_id in CAT_NAMES:
            mask = (rg['cat_id'] == cat_id)
            if mask.any():
                model = get_model_for_cat(cat_id)
                pred[mask] = np.clip(model.predict(rg.loc[mask, FEATURES].values), 0, None)
        ids.extend(rg['id'].values)
        d_nums.extend(rg['d_num'].values)
        preds.extend(pred)
        del rg
    del pf

    df_pred = pd.DataFrame({'id': ids, 'd_num': d_nums, 'pred': preds})
    del ids, d_nums, preds

    wide = df_pred.pivot(index='id', columns='d_num', values='pred')
    wide.columns = [f'F{i+1}' for i in range(PRED_HORIZON)]
    wide = wide.reset_index()
    wide['id'] = wide['id'].str.replace('_evaluation', f'_{suffix}', regex=False)
    return wide

sub_val  = predict_parquet_to_wide(VAL_PATH,  'validation')
sub_eval = predict_parquet_to_wide(EVAL_PATH, 'evaluation')

submission = pd.concat([sub_val, sub_eval], axis=0).reset_index(drop=True)
del sub_val, sub_eval

print(f'Submission shape: {submission.shape}')
print(f'Expected        : (60980, 29)')
assert submission.shape == (60980, 29), f'Shape mismatch: {submission.shape}'

OUT_SUB = str(OUTPUT_DIR / 'submission.csv')
submission.to_csv(OUT_SUB, index=False)
print(f'\nSaved: {OUT_SUB}')
print(f'Val RMSE: {rmse:.4f}')
print(f'\n=== Submission Summary ===')
print(f'Total rows: {len(submission):,} (validation: {len(submission)//2:,} + evaluation: {len(submission)//2:,})')
print(f'Columns: id + F1~F{PRED_HORIZON} ({submission.shape[1]} cols)')
print(f'F1 stats: mean={submission["F1"].mean():.2f}, std={submission["F1"].std():.2f}, min={submission["F1"].min():.2f}, max={submission["F1"].max():.2f}')
print(f'\n--- Head (5 rows) ---')
print(submission.head(5).to_string())
print(f'\n--- Tail (5 rows) ---')
print(submission.tail(5).to_string())
